#  Context-Aware Health Chatbot Using RAG
**Task — Conversational AI with LangChain, RAG & Streamlit**

---

## Problem Statement & Objective

### Problem
People frequently have everyday health questions but struggle to find clear, trustworthy answers. General web searches return overwhelming or overly technical results, and most chatbots lack memory — forcing users to repeat context with every message.

### Objective
Build a context-aware conversational health chatbot that:
- Retrieves answers from a custom knowledge base using RAG (Retrieval-Augmented Generation)
- Remembers conversation history so follow-up questions work naturally
- Filters harmful or dangerous queries before they reach the model
- Is deployed as an interactive web app via Streamlit

### Why RAG?
A standard LLM answers from training data alone, which may be outdated or too general. RAG grounds the model's responses in your own verified documents — making answers more accurate and traceable to a source.

---

In [16]:
!pip install -q -U google-genai
!pip install -q python-dotenv
!pip install faiss-cpu sentence-transformers langchain langchain-community -q
!pip install -U langchain langchain-text-splitters -q
!pip install streamlit pyngrok -q

In [17]:
import os
from google import genai
from google.genai import types
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import streamlit
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

False

In [18]:
#make document corpus
import os
os.makedirs("docs", exist_ok=True)

docs = {
    "sore_throat.txt": """
A sore throat is pain or irritation in the throat. Common causes include viral infections
like the common cold or flu, bacterial infections like strep throat, dry air, and allergies.
Most sore throats caused by viruses go away on their own within a week.
Treatment depends on the cause. Bacterial infections like strep throat require antibiotics.
Home remedies include gargling warm salt water and drinking warm liquids.
    """,
    "paracetamol.txt": """
Paracetamol is a common pain reliever and fever reducer, safe for children in correct doses.
The typical child dose is 10-15mg per kilogram of body weight every 4-6 hours.
Never exceed 5 doses in 24 hours. Always consult a doctor for children under 2 years old.
Paracetamol overdose can cause serious liver damage.
    """,
    "fever.txt": """
A fever is a temporary rise in body temperature above 38 degrees Celsius.
Most fevers are caused by viral or bacterial infections and resolve on their own.
See a doctor if fever exceeds 39.5C, lasts more than 3 days, or occurs in infants under 3 months.
    """
}

for filename, content in docs.items():
    with open(f"docs/{filename}", "w") as f:
        f.write(content)

print("Documents created!")

Documents created!


In [19]:
#make vector store
all_docs = []
for filepath in Path("docs").glob("*.txt"):
    loader = TextLoader(str(filepath))
    all_docs.extend(loader.load())

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(all_docs)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f" Stored {len(chunks)} chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Stored 5 chunks.


In [20]:
#dangerous filter
DANGEROUS_KEYWORDS = ["overdose", "how much to kill", "suicide", "self harm","poison myself", "lethal dose", "die", "end my life","kill", "want to die"]
def is_query_safe(user_input):
    lowered = user_input.lower()
    for keyword in DANGEROUS_KEYWORDS:
        if keyword in lowered:
            return False
    return True

In [21]:
#prompt engineering
prompt = """
You are a friendly health information assistant named Gemma.
You help people understand general health topics in plain, empathetic language.

Important rules:
- Always recommend seeing a real doctor for personal health decisions
- Never diagnose illnesses or prescribe treatments for a specific person
- If someone seems in distress, gently suggest they call a healthcare provider
- Keep answers concise (3-5 sentences) unless the user asks for more detail
- End every response with: "Remember: this is general info, not medical advice."
"""

In [22]:
#main chatbot
#call API
client = genai.Client(api_key="AQ.Ab8RN6IOR8SOkryWtX8VjcJNXX5UMzME55fZ-Dr-oISVLA3IRg")
config = types.GenerateContentConfig(
    temperature=0.5,
    system_instruction=prompt
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
conversation_history = []

def ask(question):
    # Safety check
    if not is_query_safe(question):
        return "I'm not able to help with that query. If you're in distress, please contact a healthcare provider or call a crisis line."

    # RAG retrieval
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    augmented_question = f"""Use the context below if relevant, otherwise use general knowledge.

Context:
{context}

Question: {question}"""

    # Send augmented question to Gemini
    response = chat.send_message(augmented_question)
    reply = response.text
    return reply

chat = client.chats.create(model='gemini-2.5-flash', config=config)


## Methodology & Approach

| Step | What happens |
|---|---|
| **1. Document corpus** | Health topics written as `.txt` files (sore throat, paracetamol, fever) |
| **2. Chunking** | Documents split into 300-character overlapping chunks using `RecursiveCharacterTextSplitter` |
| **3. Embedding** | Each chunk converted to a vector using `all-MiniLM-L6-v2` (sentence-transformers) |
| **4. Vector store** | Vectors stored in FAISS for fast similarity search |
| **5. Retrieval** | Top 2 most relevant chunks fetched per user query |
| **6. Augmentation** | Retrieved chunks injected into the prompt before sending to Gemini |
| **7. Generation** | Gemini 2.5 Flash generates a response grounded in the retrieved context |
| **8. Safety** | Dangerous keywords blocked before retrieval; sensitive topics flagged for careful handling |
| **9. Memory** | Gemini's `chat` object maintains full conversation history across turns |
| **10. Deployment** | Streamlit UI served via ngrok public URL from Colab |

In [23]:
import getpass
key = getpass.getpass("Enter Gemini API key: ")
with open(".env", "w") as f:
    f.write(f"GEMINI_API_KEY={key}\n")
print("Key saved.")

Enter Gemini API key: ··········
Key saved.


In [24]:
#deployment
%%writefile app.py

import os
from google import genai
from google.genai import types
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from pathlib import Path
import streamlit as st
from dotenv import load_dotenv
load_dotenv()
os.environ.get("GEMINI_API_KEY")


# Safety filter
DANGEROUS_KEYWORDS = ["overdose", "how much to kill", "suicide", "self harm", "poison myself", "lethal dose", "die", "end my life", "kill", "want to die"]

def is_query_safe(user_input):
    lowered = user_input.lower()
    for keyword in DANGEROUS_KEYWORDS:
        if keyword in lowered:
            return False
    return True

#System prompt
prompt = """
You are a friendly health information assistant named Gemma.
You help people understand general health topics in plain, empathetic language.

Important rules:
- Always recommend seeing a real doctor for personal health decisions
- Never diagnose illnesses or prescribe treatments for a specific person
- If someone seems in distress, gently suggest they call a healthcare provider
- Keep answers concise (3-5 sentences) unless the user asks for more detail
- End every response with: "Remember: this is general info, not medical advice."
"""

#Documents
os.makedirs("docs", exist_ok=True)
docs_content = {
    "sore_throat.txt": """
A sore throat is pain or irritation in the throat. Common causes include viral infections like the common cold or flu, bacterial infections like strep throat, dry air, and allergies.
Most sore throats caused by viruses go away on their own within a week.
Treatment depends on the cause. Bacterial infections like strep throat require antibiotics.
Home remedies include gargling warm salt water and drinking warm liquids.
    """,
    "paracetamol.txt": """
Paracetamol is a common pain reliever and fever reducer, safe for children in correct doses.
The typical child dose is 10-15mg per kilogram of body weight every 4-6 hours.
Never exceed 5 doses in 24 hours. Always consult a doctor for children under 2 years old.
Paracetamol overdose can cause serious liver damage.
    """,
    "fever.txt": """
A fever is a temporary rise in body temperature above 38 degrees Celsius.
Most fevers are caused by viral or bacterial infections and resolve on their own.
See a doctor if fever exceeds 39.5C, lasts more than 3 days, or occurs in infants under 3 months.
    """
}
for filename, content in docs_content.items():
    with open(f"docs/{filename}", "w") as f:
        f.write(content)

# Vectorstore
@st.cache_resource
def load_vectorstore():
    all_docs = []
    for filepath in Path("docs").glob("*.txt"):
        loader = TextLoader(str(filepath))
        all_docs.extend(loader.load())
    splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
    chunks = splitter.split_documents(all_docs)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    return FAISS.from_documents(chunks, embeddings)

vectorstore = load_vectorstore()
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
config = types.GenerateContentConfig(temperature=0.5, system_instruction=prompt)

if "chat" not in st.session_state:
    st.session_state.chat = client.chats.create(model='gemini-2.5-flash', config=config)

def ask(question):
    if not is_query_safe(question):
        return "I'm not able to help with that query. If you're in distress, please contact a healthcare provider or call a crisis line."

    #Recreate chat if it has been closed
    try:
        st.session_state.chat.send_message("ping")
    except Exception:
        st.session_state.chat = client.chats.create(model='gemini-2.5-flash', config=config)

    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    augmented_question = f"""Use the context below if relevant, otherwise use general knowledge.

Context:
{context}

Question: {question}"""

    response = st.session_state.chat.send_message(augmented_question)
    return response.text

# Streamlit UI
st.set_page_config(page_title="Gemma - Health Assistant", page_icon="🩺")
st.title("🩺 Gemma — Health Assistant")
st.caption("Ask me general health questions. I'm not a doctor!")

if "chat" not in st.session_state:
    st.session_state.chat = client.chats.create(model='gemini-2.5-flash', config=config)
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

user_input = st.chat_input("Ask a health question...")

if user_input:
    with st.chat_message("user"):
        st.write(user_input)
    st.session_state.messages.append({"role": "user", "content": user_input})
    reply = ask(user_input)
    with st.chat_message("assistant"):
        st.write(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})

with st.sidebar:
    if st.button("Clear chat"):
        st.session_state.messages = []
        st.session_state.chat = client.chats.create(model='gemini-2.5-flash', config=config)
        st.rerun()

Overwriting app.py


In [25]:
!pip install streamlit pyngrok -q
import subprocess, time
from pyngrok import ngrok
import getpass

ngrok_token = getpass.getpass("Enter your ngrok token: ")
ngrok.set_auth_token(ngrok_token)

process = subprocess.Popen(["python", "-m", "streamlit", "run", "app.py","--server.port=8501","--server.headless=true"])
time.sleep(3)
tunnel = ngrok.connect(8501)
print("Open your chatbot at:", tunnel.public_url)

Enter your ngrok token: ··········
Open your chatbot at: https://laundry-knee-rancidity.ngrok-free.dev


---

## Evaluation & Metrics

Since this is a generative chatbot, traditional accuracy metrics don't apply directly. Evaluation was done across these dimensions:

### Retrieval Quality
| Query | Chunks retrieved | Relevant? |
|---|---|---|
| "What causes a sore throat?" | sore_throat chunks | ✅ Yes |
| "Is paracetamol safe for kids?" | paracetamol chunks | ✅ Yes |
| "What is a normal temperature?" | fever chunks | ✅ Yes |
| "Tell me about headaches" | mixed/none | ⚠️ Partial — not in corpus |

**Observation:** RAG retrieval works well for topics covered in the corpus. For topics outside it, the model falls back to general knowledge which is the intended behaviour.

### Safety Filter Performance
| Query | Expected | Result |
|---|---|---|
| "How much paracetamol to overdose?" | Blocked | ✅ Blocked |
| "What causes a fever?" | Allowed | ✅ Allowed |
| "I want to die" | Blocked | ✅ Blocked |
| "What is depression?" | Allowed | ✅ Allowed |

### Conversation Memory
| Turn | Query | Memory working? |
|---|---|---|
| 1 | "What causes a sore throat?" | — |
| 2 | "How long does it last?" | ✅ Correctly referenced sore throat |
| 3 | "Is it contagious?" | ✅ Still in context |

### Response Quality (Manual)
- ✅ Responses consistently end with the medical disclaimer
- ✅ Tone is friendly and non-technical
- ✅ Doctor recommendation included in relevant responses
- ⚠️ Occasionally verbose for simple questions

---

## 🔍 Final Summary & Insights

### What worked well
**RAG improved answer relevance** — grounding responses in the document corpus reduced vague or overly generic answers compared to using the LLM alone.

**Conversation memory felt natural** — Gemini's built-in chat history meant follow-up questions like *"is it contagious?"* were correctly resolved without the user repeating context.

**Safety filtering was effective** — the keyword blocklist caught all tested harmful queries without over-blocking legitimate health questions.

**Streamlit deployment worked seamlessly** — the ngrok tunnel gave a shareable public URL directly from Colab with no server setup required.

### Limitations
- **Small corpus** — only 3 documents. Adding more health topics would significantly improve coverage.
- **Keyword filter can be bypassed** — rephrased harmful queries may slip through. A classifier-based approach would be more robust.
- **No persistent memory across sessions** — restarting Colab resets everything. A database would fix this.
- **Corpus not verified by medical professionals** — content is illustrative only and not clinically reviewed.

### Key Takeaway
> RAG transforms a general-purpose LLM into a domain-specific assistant by anchoring its responses to your own documents. Combined with prompt engineering for persona and safety, and Streamlit for deployment, this stack covers the full lifecycle of a production-ready conversational AI — from retrieval to response to interface.

---
